# 07.18 彩蛋：爱心动画（pygame 综合实战）

> 这一篇和 pandas 没关系，它是放在本章目录下的一个**综合实战彩蛋**。
> 前面学的**变量、循环、函数、类、数据结构、三角函数**在这里会全部用上。

### 学习目标

1. 看懂一个"图形界面程序"的基本骨架（初始化 -> 主循环 -> 收尾）
2. 理解**心形是怎么用参数方程算出来的**
3. 理解**为什么"零向量求单位向量"会直接崩溃**——这是一类真实 bug
4. 学会在 **notebook 里渲染动画帧**（不弹窗口也能看到效果）
5. 了解怎么把逐帧图片拼成一张 GIF

### 配套文件

同一目录下的 `heart_animation.py` 就是可独立运行的完整脚本：

```bash
python heart_animation.py                 # 打开窗口看动画（按 ESC 退出）
python heart_animation.py --frames 120    # 只跑 120 帧就退出（用于自动化测试）
```

**本笔记不重新写一遍代码，而是直接"调用"那个脚本里的函数**——
这是一种很实用的做法：**一份逻辑，既能当脚本跑，也能在 notebook 里拆开讲**。


## 一、这个动画做了三件事

| # | 做什么 | 用到的知识 |
|---|---|---|
| 1 | 用**参数方程**画出一个心形 | 三角函数、循环、列表 |
| 2 | 让心形按**心跳节奏**缩放 | `sin` 波、时间函数 |
| 3 | 从中心向四周喷出**粒子** | 类、随机数、向量 |

### 程序骨架：所有图形界面程序都长这样

```text
初始化（建窗口、建对象）
    ↓
while 主循环:
    1. 处理事件（键盘、鼠标、关窗）
    2. 清屏（擦掉上一帧）
    3. 计算这一帧的状态（缩放多少、粒子在哪）
    4. 画（心形 + 粒子 + 文字）
    5. 翻页显示（把刚画的这帧推给屏幕）
    6. 控制帧率
    ↓
收尾（释放资源）
```

记住这个骨架，以后你写任何小游戏/动画，都是往里填内容。


## 二、让 pygame 在 notebook 里"无窗口"运行

pygame 默认要开一个真实窗口，而 notebook（尤其是服务器/无显示器的环境）没有窗口。
解决办法：**在 `import pygame` 之前**设置环境变量，让 SDL 用一个"假"显示驱动。

```python
os.environ['SDL_VIDEODRIVER'] = 'dummy'   # 必须在 import pygame 之前！
```

这样 pygame 照常工作、照常画图，只是不往屏幕上输出，**画完的每一帧都留在内存的 Surface 里**，
我们就能把它取出来看/存成图片。


In [ ]:
import os
import sys
import math
import glob

# 【关键】必须在 import pygame 之前设置，否则不生效
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')

# 把本 notebook 所在目录加进搜索路径，才能 import 同目录的 heart_animation
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import pygame
import heart_animation as ha          # 导入真实的脚本模块

pygame.init()                          # 初始化 pygame 各子系统

print('pygame 版本：', pygame.version.ver)
print('窗口尺寸：', ha.WIDTH, 'x', ha.HEIGHT)
print('粒子数量：', ha.PARTICLE_COUNT)
print('心形采样点数：', ha.HEART_STEPS)


## 三、心形是用数学公式算出来的

心形不是"画"出来的，而是**算出来一堆点、再用多边形连起来**。
用的是一条经典的心形参数方程（`t` 从 0 走到 2π 刚好一圈）：

$$
\begin{aligned}
x &= 16 \sin^3 t \\
y &= 13 \cos t - 5 \cos 2t - 2 \cos 3t - \cos 4t
\end{aligned}
$$

代码里对应 `heart_animation.py` 的 `heart_points()` 函数。

### 一个必须注意的坑：屏幕坐标系的 y 轴是**向下**的

| 坐标系 | 原点 | y 轴方向 |
|---|---|---|
| 数学坐标系 | 中间 | **向上** |
| 屏幕坐标系 | **左上角** | **向下** |

所以公式算出来的 `y` 要**取负号**再贴到屏幕上，否则心形会是"倒着"的。
这就是代码里那句 `-y * HEART_SCALE * scale + cy` 里负号的原因。

下面直接把点算出来画一下看看：


In [ ]:
import matplotlib

# 【同样要小心】matplotlib 默认会开一个真实的绘图窗口，而且 `plt.show()`
# 在普通脚本里会**卡在那里等你手动关掉窗口**才继续。
# 处理办法：只有"不在 Jupyter 里"的时候才切到无窗口的 Agg 后端；
# 在 Jupyter 里保持 inline，图才能直接显示在单元格下方。
try:
    get_ipython                     # Jupyter 内核里才有这个函数
except NameError:
    matplotlib.use('Agg')           # 普通脚本/自动化环境：不弹窗

import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

pts = ha.heart_points(scale=1.0, center=(ha.WIDTH // 2, ha.HEIGHT // 2))
print('一共采样了', len(pts), '个点')
print('前 3 个点：', [(round(x, 1), round(y, 1)) for x, y in pts[:3]])

xs = [p[0] for p in pts]
ys = [p[1] for p in pts]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# 左图：屏幕坐标系（y 向下）——用 invert_yaxis 模拟屏幕
axes[0].fill(xs, ys, color='crimson')
axes[0].set_title('屏幕坐标系（y 轴向下）')
axes[0].set_aspect('equal')
axes[0].invert_yaxis()          # 把 y 轴翻过来，模拟屏幕
axes[0].grid(True, alpha=0.3)

# 右图：如果忘了写负号会怎样
ys_wrong = [-y for y in ys]
axes[1].fill(xs, ys_wrong, color='steelblue')
axes[1].set_title('忘记负号 -> 心形倒过来了')
axes[1].set_aspect('equal')
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 四、重头戏：那个让程序崩溃的"零向量"bug

原始版本在 pygame 2.6.1 下会**直接崩溃**，报错是：

```text
ValueError: Can't normalize Vector of length Zero
```

### 为什么会这样？

原代码想算"从中心指向粒子的方向"，写的是：

```python
direction = (pos - center).normalize()
```

但粒子的初始位置 **`pos` 就是 `center` 本身**，两边相减得到 **零向量 `(0, 0)`**。
而 `normalize()`（单位化）要做的是"把向量长度除成 1"，
也就是 **除以自己的长度**——零向量的长度是 0，等于**除以 0**。

于是新版本 pygame 直接抛异常。

> **为什么这个 bug 藏了很久？** 因为老版本 pygame 遇到零向量会**静默返回零向量**，
> 不报错。程序能跑，但速度恒为 `(0, 0)`，结果是
> **100 个粒子全部叠在正中心一动不动**，根本看不到"喷射"效果。
> 这类"不报错但逻辑错"的问题，比直接崩溃更难发现。

下面亲手复现一下这个报错：


In [ ]:
center = pygame.Vector2(ha.WIDTH // 2, ha.HEIGHT // 2)
zero = center - pygame.Vector2(ha.WIDTH // 2, ha.HEIGHT // 2)

print('两个向量相减得到：', zero)
print('它的长度是：', zero.length())
print()

try:
    zero.normalize()          # 单位化 = 除以长度 = 除以 0
    print('居然没报错')
except ValueError as e:
    print('复现成功，pygame 直接报错：')
    print('   ValueError:', e)


### 正确的修法：不去"归一化"，直接给一个随机方向

关键洞察：**我们本来就不需要一个"从中心指向某处"的向量**——
我们想要的是"朝随机方向飞出去"。

而 `(cos θ, sin θ)` 这个组合**天生就是长度为 1 的单位向量**，
根本不需要做归一化，也就不可能除以 0：

```python
angle = random.uniform(0, math.tau)          # math.tau = 2π，随机取一整圈里的角度
direction = pygame.Vector2(math.cos(angle), math.sin(angle))
```

修完之后，粒子会朝**四面八方**均匀散开。看下面这个方向分布：


In [ ]:
import random
import math

random.seed(42)
N = ha.PARTICLE_COUNT
angles = [random.uniform(0, math.tau) for _ in range(N)]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={'projection': 'polar'})
ax.scatter(angles, [1] * N, s=18, alpha=0.7, color='crimson')
ax.set_title(f'{N} 个粒子的飞行方向分布\n（均匀铺满一整圈，不再是零向量）', pad=18)
plt.show()

print('共有', len(set(round(a, 4) for a in angles)), '个互不相同的方向')
print('每个方向向量长度都是 1 ->',
      round(pygame.Vector2(math.cos(angles[0]), math.sin(angles[0])).length(), 6))


## 五、在 notebook 里把动画"渲染"出来

不开窗口，我们照样能把每一帧画到内存 Surface 上，再取出来显示。

核心两行：

```python
surf = pygame.Surface((WIDTH, HEIGHT))      # 一张内存画布（不需要真实窗口）
raw = pygame.image.tobytes(surf, 'RGB')     # 把像素取出来，转成 PIL 图片
```


In [ ]:
from PIL import Image

# pygame 2.x 把 tostring 改名为 tobytes，这里做个兼容
_to_bytes = getattr(pygame.image, 'tobytes', None) or pygame.image.tostring


def render_frame(scale, particles):
    """把某一帧画到内存画布上，返回 PIL Image（不弹窗口）。"""
    surf = pygame.Surface((ha.WIDTH, ha.HEIGHT))
    surf.fill(ha.BG_COLOR)

    # 画心形：先填充，再描边（和脚本里完全一致）
    pts = ha.heart_points(scale, (ha.WIDTH // 2, ha.HEIGHT // 2))
    pygame.draw.polygon(surf, ha.RED, pts)
    pygame.draw.lines(surf, ha.PINK, True, pts, 3)

    # 更新并绘制所有粒子
    for p in particles:
        p.update()
        p.draw(surf)

    raw = _to_bytes(surf, 'RGB')
    return Image.frombytes('RGB', (ha.WIDTH, ha.HEIGHT), raw)


# 先看单个心跳周期里的 4 个瞬间（心形大小随 sin 变化）
particles = [ha.Particle((ha.WIDTH // 2, ha.HEIGHT // 2)) for _ in range(ha.PARTICLE_COUNT)]
stills = []
for step in range(4):
    phase = step * math.pi / 2                              # 0, π/2, π, 3π/2
    s = 1.0 + ha.PULSE_AMPLITUDE * math.sin(phase)
    stills.append((phase, render_frame(s, particles)))

fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, (phase, img) in zip(axes, stills):
    ax.imshow(img)
    ax.set_title(f'相位 {phase:.2f}π\n缩放 {1 + ha.PULSE_AMPLITUDE * math.sin(phase):.3f}')
    ax.axis('off')
plt.tight_layout()
plt.show()


## 六、把逐帧图片拼成一张 GIF

PIL 的 `Image.save(..., save_all=True, append_images=[...])` 可以把多张图合成动图：

| 参数 | 作用 |
|---|---|
| `save_all=True` | 保存所有帧（否则只存第一帧） |
| `append_images=[...]` | 后续的帧 |
| `duration` | 每帧显示多少毫秒（越小越快） |
| `loop=0` | 无限循环播放 |


In [ ]:
OUT_DIR = os.path.join(os.getcwd(), '生成文件')
os.makedirs(OUT_DIR, exist_ok=True)

# 重新造一批粒子，从头开始渲染一个完整循环
random.seed(7)
particles = [ha.Particle((ha.WIDTH // 2, ha.HEIGHT // 2)) for _ in range(ha.PARTICLE_COUNT)]

FRAMES = 48
gif_frames = []
for i in range(FRAMES):
    # 让 sin 在 48 帧里正好走完 2 个完整心跳周期
    s = 1.0 + ha.PULSE_AMPLITUDE * math.sin(i / FRAMES * 2 * math.tau)
    gif_frames.append(render_frame(s, particles))

gif_path = os.path.join(OUT_DIR, '爱心动画.gif')
gif_frames[0].save(
    gif_path,
    save_all=True,
    append_images=gif_frames[1:],
    duration=40,          # 每帧 40ms
    loop=0,               # 0 = 无限循环
    optimize=True,
)
print(f'已生成动图：{gif_path}')
print(f'共 {len(gif_frames)} 帧，大小 {os.path.getsize(gif_path) / 1024:.1f} KB')


在 Jupyter 里，下面这行会把动图直接显示出来：


In [ ]:
try:
    from IPython.display import Image as IPyImage, display

    display(IPyImage(filename=gif_path))
except Exception as e:      # 非 IPython 环境下就跳过显示
    print('（当前环境不显示内嵌图片，文件已保存到上面那个路径）', type(e).__name__)


## 七、原脚本怎么跑

| 命令 | 效果 |
|---|---|
| `python heart_animation.py` | 打开窗口看动画，按 ESC 或点 × 退出 |
| `python heart_animation.py --frames 120` | 只跑 120 帧自动退出（自动化测试用） |

### 主循环里每一步在干什么

| 步骤 | 代码 | 说明 |
|---|---|---|
| 1 处理事件 | `for event in pygame.event.get()` | **必须做**，否则窗口"无响应" |
| 2 清屏 | `screen.fill(BG_COLOR)` | 相当于擦掉上一帧 |
| 3 算缩放 | `scale = 1 + A * sin(ticks / SPEED)` | 用 sin 做平滑往复，模拟心跳 |
| 4 画心 | `draw.polygon` + `draw.lines` | 先填充再描边 |
| 5 画粒子 | `p.update()` + `p.draw(screen)` | 先更新状态，再画 |
| 6 文字 | `font.render` + `screen.blit` | 字体对象只建一次 |
| 7 翻页 | `pygame.display.flip()` | 把这一帧推给屏幕 |
| 8 限速 | `clock.tick(FPS)` | **不加会跑满 CPU、风扇狂转** |

### 原脚本修掉的另外两个小问题

| 问题 | 后果 | 修法 |
|---|---|---|
| `heart_points` 算了一遍没用，主循环里又重算 | 白算一次，代码重复 | 抽成函数，复用结果 |
| 每帧都 `pygame.font.SysFont(...)` 新建字体 | 每帧都读一次字体文件，白白浪费性能 | 提到循环外，只建一次 |


## 八、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 对零向量做 `normalize()` | `ValueError: Can't normalize Vector of length Zero` | 改用随机方向角 `(cos θ, sin θ)` |
| 忘了处理事件 | 窗口白屏、显示"无响应" | 主循环里必须有 `pygame.event.get()` |
| 忘记了 `screen.fill()` | 画面拖影、越画越脏 | 每帧先清屏 |
| 忘了 `clock.tick(FPS)` | CPU 占满、风扇狂转 | 每帧限速 |
| 没设 `SDL_VIDEODRIVER=dummy` 就 import pygame | notebook 里起不来窗口 | 环境变量必须在 import **之前**设 |
| 在普通脚本里直接 `plt.show()` | **弹出窗口并卡住**，等你去关它 | 设 `MPLBACKEND=Agg`，或 `matplotlib.use('Agg')` |
| 装完没 `pygame.quit()` | 资源不释放，反复运行可能出错 | 结束时调用 |
| 屏幕 y 轴方向搞反 | 图形上下颠倒 | 记住屏幕原点在左上角，`y` 要取负 |


## 九、动手练习

1. **改颜色**：把心形的填充色从红色改成金色，重新生成 GIF 看看
2. **改节奏**：调 `PULSE_SPEED`，让心跳变快/变慢，观察区别
3. **改粒子**：把粒子改成圆形（提示：`pygame.draw.circle`），并把数量调到 300
4. **换形状**：把心形参数方程换成其他曲线（比如玫瑰线 `r = cos(3t)`），画出新图形
5. **加文字**：在 `render_frame` 里加上 `pygame.font` 渲染的文字，让 GIF 里也显示标题
6. **反着来**：故意把 `zero.normalize()` 那段的异常吞掉（`try/except` 不打印），
   体会一下"不报错的 bug 有多难查"

---

**答案思路（先自己做）**

```python
# 1. 改颜色：只改配置区的常量即可
ha.RED = (255, 215, 0)          # 金色填充

# 3. 圆形粒子：改 heart_animation.py 里 Particle.draw
#    pygame.draw.circle(surface, self.color,
#                       (int(self.pos.x), int(self.pos.y)), max(1, int(self.size)))

# 4. 玫瑰线
#    x = r * cos(t), y = r * sin(t)，其中 r = cos(3t)
```


In [ ]:
# 练习 1 小演示：把填充色改掉，再渲染一帧看看效果
original_red = ha.RED
ha.RED = (255, 215, 0)          # 金色

demo_particles = [ha.Particle((ha.WIDTH // 2, ha.HEIGHT // 2)) for _ in range(40)]
demo_img = render_frame(1.02, demo_particles)

ha.RED = original_red           # 【好习惯】改完记得还原，避免影响后面的代码

plt.figure(figsize=(6, 4.5))
plt.imshow(demo_img)
plt.title('练习 1：换成金色填充（渲染一帧）')
plt.axis('off')
plt.show()
